# Part 5: More replicas, and where each request goes

Fill both routers first (`pytest tests/test_routing.py`). The runs, from the repo root, one at a time:

```bash
# how many replicas: Sarathi on 2 replicas, round-robin (about 5 minutes)
REPLICAS=2 GLOBAL=round_robin SCHED=sarathi CHUNK=512 QPS=5.53 bash scripts/run_default.sh traces/azure_conversation.csv

# routers on the long-short bursts trace (seconds each)
for G in round_robin lor lot; do
  REPLICAS=2 GLOBAL=$G SCHED=sarathi CHUNK=512 QPS=8 bash scripts/run_default.sh traces/long_short_bursts.csv
done

# LOT on the real trace (about 5 minutes)
REPLICAS=2 GLOBAL=lot SCHED=sarathi CHUNK=512 QPS=5.53 bash scripts/run_default.sh traces/azure_conversation.csv
```

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "traces").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analysis import workload as wl
from analysis.replay import busy_percent, describe_run, load_run
from analysis.workload import load_trace, predict_queue_wait

azure = load_trace(ROOT / "traces" / "azure_conversation.csv")
bursts = load_trace(ROOT / "traces" / "long_short_bursts.csv")

def summary(run_dir, trace):
    run = load_run(run_dir, trace)
    tbt = run["tbt"].set_index("percentile").seconds
    req = run["requests"]
    return {"wait_p99": req.wait.quantile(0.99), "e2e_p99": req.e2e.quantile(0.99),
            "tbt_p99_ms": tbt[99] * 1e3, "busy": busy_percent(run_dir)}

## How many replicas?

In [2]:
for n in (1, 2, 3):
    w = predict_queue_wait(azure, prefill_tok_per_s=n * wl.PREFILL_TOK_PER_S,
                           decode_tok_per_s=n * wl.DECODE_TOK_PER_S)
    print(f"{n} replica(s): predicted P99 wait {np.quantile(w, 0.99):.2f} s")

ONE_REPLICA = ROOT / "simulator_output" / "C512"   # from Part 3
TWO_RR = ROOT / "simulator_output" / "AZURE_2_ROUND_ROBIN"
for d in (ONE_REPLICA, TWO_RR):
    s = summary(d, azure)
    print(f"{describe_run(d):42s} P99 wait {s['wait_p99']:6.2f} s   P99 time between tokens {s['tbt_p99_ms']:.0f} ms")

1 replica(s): predicted P99 wait 72.13 s
2 replica(s): predicted P99 wait 0.38 s
3 replica(s): predicted P99 wait 0.15 s
sarathi C=512                              P99 wait 171.16 s   P99 time between tokens 62 ms
sarathi C=512, 2 replicas, round_robin     P99 wait   0.60 s   P99 time between tokens 54 ms


## Routers on the long-short bursts trace

In [3]:
BURST_RUNS = [ROOT / "simulator_output" / d for d in [
    "BURSTS_ROUND_ROBIN", "BURSTS_LOR", "BURSTS_LOT",
]]
for d in BURST_RUNS:
    s = summary(d, bursts)
    print(f"{describe_run(d):42s} wait P99 {s['wait_p99']:5.2f} s   end-to-end P99 {s['e2e_p99']:5.2f} s   "
          f"busy % (replica 1 / 2) {' / '.join(f'{b:.0f}' for b in s['busy'])}")

sarathi C=512, 2 replicas, round_robin     wait P99  8.99 s   end-to-end P99  9.78 s   busy % (replica 1 / 2) 100 / 5
sarathi C=512, 2 replicas, lor             wait P99  1.14 s   end-to-end P99  1.98 s   busy % (replica 1 / 2) 80 / 76
sarathi C=512, 2 replicas, lot             wait P99  0.77 s   end-to-end P99  0.85 s   busy % (replica 1 / 2) 81 / 81


## Round-robin vs LOT on the real trace

In [4]:
TWO_LOT = ROOT / "simulator_output" / "AZURE_2_LOT"
for d in (TWO_RR, TWO_LOT):
    s = summary(d, azure)
    print(f"{describe_run(d):42s} P99 wait {s['wait_p99']:5.2f} s   P99 time between tokens {s['tbt_p99_ms']:.0f} ms")

sarathi C=512, 2 replicas, round_robin     P99 wait  0.60 s   P99 time between tokens 54 ms
sarathi C=512, 2 replicas, lot             P99 wait  0.40 s   P99 time between tokens 54 ms


## Questions

**Q1.** Fill in the table. What is the smallest number of replicas that meets both
targets (P99 wait under 2 s, P99 time between tokens under 100 ms)? How
close was your Part 1 model to Vidur?

| Replicas | Predicted P99 wait (Part 1 model) | Vidur P99 wait | Vidur P99 time between tokens |
|---|---|---|---|
| 1 | 72.13s | 171.16s | 62ms |
| 2 | 0.38s | 0.60s | 54ms |
| 3 | 0.15s | — | — |

**The smallest number of replicas that meets both targets is 2 replicas, with a P99 wait of 0.60s and P99 time between tokens of 54ms. For 2 replicas, the Part 1 model was similar to Vidur, with a 0.22s increase in time. There was a large difference for 1 replica with about 100s added for the Vidur P99 wait time.**

**Q2.** Fill in the table for the long-short bursts trace. Which router balances the
two replicas best, and which worst? Explain each router's busy split and wait. The worked example shows one burst on empty replicas: what changes when later bursts arrive while earlier requests are still outstanding? (Replicas 1 / 2 in the output are replica ids 0 / 1.)

| Router | Wait P99 | End-to-end P99 | Busy % (replica 1 / 2) |
|---|---|---|---|
| Round-robin | 8.99s | 9.78s | 100 / 5 |
| LOR | 1.14s | 1.98s | 80 / 76 |
| LOT | 0.77s | 0.85s | 81 / 81 |

**LOT balances the two replicas the best with 81% / 81% busy and the lowest P99 wait of 0.77s. Round robin balances the two replicas the worst with 100% / 5% busy and the highest P99 wait of 8.99s since it does not consider request sizes and has 1 overused replica and 1 largely idle replica. LOR has an almost even split with 80% / 76% busy and a low P99 wait of 1.98s since it has equal request counts not necessarily equal work division. When later bursts arrive while earlier requests are still outstanding, the replicas could have different amounts of work not completed. LOR incorporates earlier outstanding requests and LOT includes their remaining tokens so they adjust their assignments based on the existing load. However, round robin does not take the existing load into consideration and will continue to rotate through each.**

**Q3.** Vidur's original LOR counted only requests **waiting** in each replica's
queue, not running ones. On the real trace with 2 replicas, those queues are
almost always empty. Which replica would that router pick for almost every
request, and why? Which definition is better, and why?

**The original LOR would pick replica 1 (id=0) for almost every request because both replica's waiting queues were typically empty, which caused a tie that lead to the lower replica id to be picked. Accounting for all outstanding requests that are waiting and running is better because even though a replica has an empty queue, it may still be processing requests, so their inclusion could reduce overloading.**

**Q4.** On the real trace with 2 replicas, compare round-robin with LOT: P99 wait
and P99 time between tokens. Does the router matter here? In one sentence:
when is a load-aware router worth it?

**Round-robin's P99 wait of 0.60s was longer than LOT's P99 wait of 0.40s, and both shared the same P99 time between tokens of 54ms. Since both routers meet the targets of P99 wait of less than 2s and P99 time between token of less than 100ms, the router does not matter here. A load-aware router is worth it when the request sizes are not the same or when bursts create uneven distribution of outstanding work across replicas.**